In [ ]:
from pathlib import Path
import pandas as pd
import numpy as np

ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
DATA_DIR = ROOT / "data" / "clean"
EXPORT_DIR = ROOT / "exports"
EXPORT_DIR.mkdir(parents=True, exist_ok=True)

payments = pd.read_csv(
    DATA_DIR / "fact_payments_monthly.csv",
    parse_dates=["date"]
)

upi = pd.read_csv(
    DATA_DIR / "upi_monthly_detail.csv",
    parse_dates=["date"]
)

dates = pd.read_csv(
    DATA_DIR / "dim_date.csv",
    parse_dates=["date"]
)

print("Payments:", payments.shape)
print("UPI detail:", upi.shape)
print("Date dimension:", dates.shape)
print("Date range:", payments["date"].min(), "to", payments["date"].max())
print("Payment methods:", sorted(payments["payment_method"].unique()))

display(payments.head())
display(upi.head())

In [ ]:
# Load the datasets
payments = pd.read_csv(
    DATA_DIR / "fact_payments_monthly.csv",
    parse_dates=["date"]
)

upi = pd.read_csv(
    DATA_DIR / "upi_monthly_detail.csv",
    parse_dates=["date"]
)

dates = pd.read_csv(
    DATA_DIR / "dim_date.csv",
    parse_dates=["date"]
)

# Display dataset dimensions
print("Payments dataset:", payments.shape)
print("UPI dataset:", upi.shape)
print("Date dimension:", dates.shape)

# Preview each dataset
print("\n--- Payments Data ---")
display(payments.head())

print("\n--- UPI Details ---")
display(upi.head())

print("\n--- Date Dimension ---")
display(dates.head())

In [ ]:
# 1. Check column names and data types
print("PAYMENTS DATA TYPES")
print(payments.dtypes)

# 2. Check missing values
print("\nMISSING VALUES")
print("Payments:\n", payments.isnull().sum())
print("\nUPI:\n", upi.isnull().sum())
print("\nDates:\n", dates.isnull().sum())

# 3. Check duplicate records
print("\nDUPLICATES")
print(
    "Payments:",
    payments.duplicated(["date", "payment_method"]).sum()
)
print("UPI:", upi.duplicated(["date"]).sum())
print("Dates:", dates.duplicated(["date"]).sum())

# 4. Check date ranges
print("\nDATE RANGES")
print("Payments:", payments["date"].min(), "to", payments["date"].max())
print("UPI:", upi["date"].min(), "to", upi["date"].max())
print("Dates:", dates["date"].min(), "to", dates["date"].max())

# 5. Check payment methods
print("\nPAYMENT METHODS")
print(sorted(payments["payment_method"].unique()))

# 6. Check for negative transaction values
print("\nNEGATIVE VALUES")
print(
    (payments[["volume_mn", "value_cr"]] < 0).sum()
)

In [ ]:
print("Duplicate payment records:",
      payments.duplicated(["date", "payment_method"]).sum())

print("Duplicate UPI records:",
      upi.duplicated(["date"]).sum())

print("Duplicate date-dimension records:",
      dates.duplicated(["date"]).sum())

print("\nDate ranges:")
print("Payments:", payments["date"].min(), "to", payments["date"].max())
print("UPI:", upi["date"].min(), "to", upi["date"].max())
print("Date dimension:", dates["date"].min(), "to", dates["date"].max())

print("\nPayment methods:")
print(sorted(payments["payment_method"].unique()))

In [ ]:

# Sort and validate the cleaned payment data
payments = payments.sort_values(
    ["payment_method", "date"]
).copy()

assert not payments.duplicated(["date", "payment_method"]).any()
assert payments[["volume_mn", "value_cr"]].notna().all().all()
assert (payments[["volume_mn", "value_cr"]] >= 0).all().all()

# Monthly totals across all payment methods
monthly_totals = (
    payments.groupby("date", as_index=False)
    .agg(
        total_volume_mn=("volume_mn", "sum"),
        total_value_cr=("value_cr", "sum")
    )
)

payments = payments.merge(monthly_totals, on="date", how="left")

# Month-over-month and year-over-year growth, by payment method
grouped = payments.groupby("payment_method", group_keys=False)

payments["mom_vol_pct"] = grouped["volume_mn"].pct_change() * 100
payments["yoy_vol_pct"] = grouped["volume_mn"].pct_change(12) * 100
payments["yoy_val_pct"] = grouped["value_cr"].pct_change(12) * 100

# Share of total monthly payment volume and value
payments["share_vol_pct"] = (
    payments["volume_mn"] / payments["total_volume_mn"] * 100
)
payments["share_val_pct"] = (
    payments["value_cr"] / payments["total_value_cr"] * 100
)

# Average ticket size in INR
payments["avg_ticket_inr"] = (
    payments["value_cr"] * 1e7
    / (payments["volume_mn"] * 1e6)
).where(payments["volume_mn"] > 0)

# Flag unusually large month-over-month volume changes
payments["anomaly_reason"] = ""
payments["is_anomaly"] = False

for method, indices in payments.groupby("payment_method").groups.items():
    growth = payments.loc[indices, "mom_vol_pct"]
    mean = growth.mean()
    std = growth.std()

    if pd.notna(std) and std > 0:
        flagged = (growth - mean).abs() > 3 * std
        payments.loc[indices, "is_anomaly"] = flagged.fillna(False)
        payments.loc[
            indices[flagged.fillna(False)], "anomaly_reason"
        ] = "MoM volume growth exceeds 3 standard deviations"

# Trend classification
payments["trend_label"] = np.select(
    [
        payments["mom_vol_pct"] > 1,
        payments["mom_vol_pct"] < -1
    ],
    ["Increasing", "Decreasing"],
    default="Stable"
)

# Keep the established export schema
metric_columns = [
    "date", "payment_method",
    "mom_vol_pct", "yoy_vol_pct", "yoy_val_pct",
    "share_vol_pct", "share_val_pct", "avg_ticket_inr",
    "is_anomaly", "anomaly_reason", "trend_label"
]

eda_metrics = payments[metric_columns].copy()

display(eda_metrics.head(10))
print("Metrics shape:", eda_metrics.shape)
print("Anomalies flagged:", int(eda_metrics["is_anomaly"].sum()))


In [ ]:
# Check whether payment-method shares total 100% each month
volume_share_check = payments.groupby("date")["share_vol_pct"].sum()
value_share_check = payments.groupby("date")["share_val_pct"].sum()

print("Volume shares sum to 100%:",
      np.allclose(volume_share_check, 100))

print("Value shares sum to 100%:",
      np.allclose(value_share_check, 100))

print("\nMissing growth values:")
print(payments[["mom_vol_pct", "yoy_vol_pct", "yoy_val_pct"]].isna().sum())

print("\nAverage ticket calculation check:")
print(
    "Maximum absolute difference:",
    (
        payments["avg_ticket_inr"]
        - payments["value_cr"] * 10
          / payments["volume_mn"]
    ).abs().max()
)

In [ ]:
# Initialize required output columns
payments["is_anomaly"] = False
payments["anomaly_reason"] = ""
payments["trend_label"] = "Stable"

for method, group in payments.groupby("payment_method"):
    growth = group["mom_vol_pct"]

    mean_growth = growth.mean()
    std_growth = growth.std()

    # Flag unusually large or small monthly growth rates
    if pd.notna(std_growth) and std_growth > 0:
        anomaly_mask = (
            (growth - mean_growth).abs() > 3 * std_growth
        )
        payments.loc[group.index, "is_anomaly"] = (
            anomaly_mask.fillna(False)
        )

        flagged_indices = group.index[anomaly_mask.fillna(False)]
        payments.loc[flagged_indices, "anomaly_reason"] = (
            "MoM volume growth exceeds 3 standard deviations"
        )

    # Label the direction of monthly volume growth
    payments.loc[group.index, "trend_label"] = np.select(
        [
            growth > 1,
            growth < -1
        ],
        [
            "Increasing",
            "Decreasing"
        ],
        default="Stable"
    )

# Display the results
print("Anomalies detected:", payments["is_anomaly"].sum())

display(
    payments.loc[
        payments["is_anomaly"],
        ["date", "payment_method", "mom_vol_pct",
         "is_anomaly", "anomaly_reason", "trend_label"]
    ]
)

print("\nTrend-label counts:")
print(payments["trend_label"].value_counts())

print("\nSample output:")
display(
    payments[
        ["date", "payment_method", "mom_vol_pct",
         "is_anomaly", "anomaly_reason", "trend_label"]
    ].tail(12)
)

In [ ]:
# Required column order from the project contract
eda_columns = [
    "date",
    "payment_method",
    "mom_vol_pct",
    "yoy_vol_pct",
    "yoy_val_pct",
    "share_vol_pct",
    "share_val_pct",
    "avg_ticket_inr",
    "is_anomaly",
    "anomaly_reason",
    "trend_label"
]

# Create the final EDA export
eda_metrics = payments[eda_columns].copy()

# Format dates consistently
eda_metrics["date"] = eda_metrics["date"].dt.strftime("%Y-%m-%d")

# Save CSV
eda_path = EXPORT_DIR / "eda_metrics.csv"
eda_metrics.to_csv(eda_path, index=False)

print("Saved to:", eda_path)
print("Shape:", eda_metrics.shape)
print("\nColumns:")
print(eda_metrics.columns.tolist())

print("\nAnomalies detected:", eda_metrics["is_anomaly"].sum())
print("\nPreview:")
display(eda_metrics.head(10))

In [ ]:
# Work with the original payments data and calculated metrics
monthly = payments.groupby("date").agg(
    total_volume_mn=("volume_mn", "sum"),
    total_value_cr=("value_cr", "sum")
).sort_index()

upi_data = payments[payments["payment_method"] == "UPI"].sort_values("date")

# Calculate UPI's share of total volume and value
upi_volume_share = upi_data["share_vol_pct"]
upi_value_share = upi_data["share_val_pct"]

start_volume_share = upi_volume_share.iloc[0]
end_volume_share = upi_volume_share.iloc[-1]

start_value_share = upi_value_share.iloc[0]
end_value_share = upi_value_share.iloc[-1]

# Calculate total UPI volume growth across the sample period
upi_start_volume = upi_data["volume_mn"].iloc[0]
upi_end_volume = upi_data["volume_mn"].iloc[-1]

upi_volume_growth = (
    (upi_end_volume / upi_start_volume) - 1
) * 100

# Calculate average ticket size change
ticket_start = upi_data["avg_ticket_inr"].iloc[0]
ticket_end = upi_data["avg_ticket_inr"].iloc[-1]

ticket_change = ticket_end - ticket_start

# Build the insights table
insights = pd.DataFrame([
    {
        "order": 1,
        "stage": "hook",
        "headline": "UPI's transaction volume share",
        "detail": (
            "Tracks UPI's share of total transaction volume across "
            "the study period using the cleaned payment dataset."
        ),
        "metric_value": round(end_volume_share, 2),
        "metric_label": "UPI volume share at end (%)"
    },
    {
        "order": 2,
        "stage": "buildup",
        "headline": "UPI volume growth",
        "detail": (
           "Measures the percentage change in UPI transaction volume "
           "between the first and last months of the study period."
        ),
        "metric_value": round(upi_volume_growth, 2),
        "metric_label": "UPI volume growth over sample (%)"
    },
    {
        "order": 3,
        "stage": "insight",
        "headline": "UPI's transaction value share",
        "detail": (
           "Tracks UPI's share of the combined transaction value "
           "across the payment methods included in the dataset."
        ),
        "metric_value": round(end_value_share, 2),
        "metric_label": "UPI value share at end (%)"
    },
    {
        "order": 4,
        "stage": "conclusion",
        "headline": "UPI average ticket size",
        "detail": (
            "Measures the change in UPI's average transaction value "
            "between the first and last months of the study period."        
        ),
        "metric_value": round(ticket_change, 2),
        "metric_label": "Change in average ticket size (INR)"
    }
])

insights_path = EXPORT_DIR / "key_insights.csv"
insights.to_csv(insights_path, index=False)

print("Saved to:", insights_path)
display(insights)

In [ ]:
# Reload the saved files to validate the actual CSVs
eda_check = pd.read_csv(EXPORT_DIR / "eda_metrics.csv")
insights_check = pd.read_csv(EXPORT_DIR / "key_insights.csv")

expected_eda_columns = [
    "date", "payment_method", "mom_vol_pct", "yoy_vol_pct",
    "yoy_val_pct", "share_vol_pct", "share_val_pct",
    "avg_ticket_inr", "is_anomaly", "anomaly_reason", "trend_label"
]

expected_insight_columns = [
    "order", "stage", "headline", "detail",
    "metric_value", "metric_label"
]

print("EDA columns correct:",
      eda_check.columns.tolist() == expected_eda_columns)

print("EDA row count correct:", len(eda_check) == 168)

print("Insight columns correct:",
      insights_check.columns.tolist() == expected_insight_columns)

print("Insight stages correct:",
      insights_check["stage"].tolist() ==
      ["hook", "buildup", "insight", "conclusion"])

print("EDA contains missing dates:", eda_check["date"].isna().any())
print("Insight metrics missing:", insights_check["metric_value"].isna().any())

print("\nExport files:")
print((EXPORT_DIR / "eda_metrics.csv").resolve())
print((EXPORT_DIR / "key_insights.csv").resolve())